# Phase 3.7 - Expert Data Collection (PI Controller)
This notebook runs the Decentralized PI Controller in parallel over 200 randomized virtual patients to collect `(observation, action)` pairs for DRL Imitation Warm-starting (Phase 5).

**Why PI over Starling?**
- PI achieves better flow balance (17-25 mL/s vs 28-34 mL/s Starling)
- PI has lower congestion rate (5.9% vs 14.7% Starling)
- PI has better pressure tracking (IAE) in most scenarios
- PI runs at the same speed as Starling (~23s/episode) since both are algebraic controllers and the bottleneck is the environment's ODE solver

**Episode Budget:** 200 episodes × 2000 steps = 400,000 transitions, sufficient for behavioral cloning on a 9-input/2-output network.

**Pre-requisite:** Make sure you have uploaded `bivad_env.py`, the `simulator/` folder, and the `controllers/` folder to the Colab file explorer before running.

In [ ]:
!pip install gymnasium stable-baselines3 scipy numpy tqdm

In [ ]:
import numpy as np
import multiprocessing as mp
from google.colab import files
from tqdm.notebook import tqdm

# Import your modules
from bivad_env import BiVADEnv
from controllers.pi_controller import PIDecentralizedController

def collect_single_episode(seed):
    env = BiVADEnv(randomize=True)
    controller = PIDecentralizedController()
    
    observations = []
    actions = []
    
    obs, _ = env.reset(seed=seed)
    if hasattr(controller, 'reset'):
        controller.reset()
    
    info = {
        'LVAD_RPM': env.params['LVAD_RPM'], 'RVAD_RPM': env.params['RVAD_RPM'],
        'true_state': env.state, 'env_params': env.params
    }
    
    terminated = False
    while not terminated:
        action = controller.get_action(obs, info)
        observations.append(obs)
        actions.append(action)
        obs, reward, terminated, truncated, info = env.step(action)
            
    return observations, actions

def collect_expert_data(num_episodes=200, filename='expert_data_pi.npz'):
    print(f"Starting parallel PI data collection for {num_episodes} episodes across {mp.cpu_count()} CPU cores...")
    seeds = [np.random.randint(0, 1000000) for _ in range(num_episodes)]
    
    all_obs = []
    all_acts = []
    
    with mp.Pool(processes=mp.cpu_count()) as pool:
        for obs, acts in tqdm(pool.imap(collect_single_episode, seeds), total=num_episodes, desc="Episodes Completed"):
            all_obs.extend(obs)
            all_acts.extend(acts)
        
    all_obs = np.array(all_obs)
    all_acts = np.array(all_acts)
    
    np.savez(filename, obs=all_obs, acts=all_acts)
    print(f"\nSaved {len(all_acts)} expert transitions to {filename}.")
    print(f"Obs shape: {all_obs.shape}, Acts shape: {all_acts.shape}")
    
    print("Triggering download...")
    files.download(filename)

# Start collection (200 episodes)
collect_expert_data(num_episodes=200)
